# 03 — LoRA SFT Training

Train LoRA adapters on filtered SynSQL data using Unsloth + TRL.
Grid search over rank, target modules, learning rate, RSLoRA.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def bootstrap_repo():
    configured = os.environ.get('SLM_PRJ_PATH')
    candidates = [Path(configured).expanduser()] if configured else []
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents,
                       Path('/kaggle/working/SLM-PRJ'),
                       Path('/kaggle/working/SLMSQL_project')])
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'benchmarks').is_dir():
            return candidate
    target = Path('/kaggle/working/SLM-PRJ')
    url = os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git')
    subprocess.run(['git', 'clone', url, str(target)], check=True)
    return target

REPO_DIR = bootstrap_repo()
sys.path.insert(0, str(REPO_DIR))
from scripts.kaggle_utils import prepare_repo_import, resolve_data_dir
REPO_DIR = prepare_repo_import(REPO_DIR)
DATA_DIR = resolve_data_dir(REPO_DIR)

RESULTS_DIR = REPO_DIR / 'results' / 'lora'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
from scripts.kaggle_utils import load_kaggle_secrets, publish_checkpoint
from training.lora_config import generate_experiments
from training.sft_trainer import train
from training.merge_adapter import merge_and_save
print(f'Repo: {REPO_DIR}')
print(f'Data: {DATA_DIR}')

## Grid Search on 5K Subset

In [ ]:
MODEL_KEY = os.environ.get('LORA_MODEL_KEY', 'small')
experiments = generate_experiments('configs/lora_grid.yaml', MODEL_KEY)
max_experiments = int(os.environ.get('MAX_EXPERIMENTS', '1'))
if os.environ.get('RUN_FULL_GRID', '0') != '1':
    experiments = experiments[:max_experiments]
print(f'{len(experiments)} experiment(s) selected (set RUN_FULL_GRID=1 for the full grid)')
for e in experiments:
    print(f'  {e.name}: rank={e.rank}, modules={e.target_modules}, lr={e.learning_rate}, rslora={e.rslora}')


In [ ]:
TRAIN_DATA = Path(os.environ.get('TRAIN_DATA', str(DATA_DIR / 'sft_subsets' / 'sft_10000.jsonl')))
if not TRAIN_DATA.is_file():
    raise FileNotFoundError(
        f'No SFT data at {TRAIN_DATA}. Run 01_data_prep after attaching '
        'SynSQL/Syn Think, or set TRAIN_DATA to an existing JSONL.'
    )

resume_from = os.environ.get('RESUME_FROM')
grid_results = []
for exp in experiments:
    print(f'\n=== Training {exp.name} ===')
    start = time.time()
    output_dir = RESULTS_DIR / exp.name
    try:
        result = train(
            exp,
            TRAIN_DATA,
            output_dir,
            resume_from=resume_from if resume_from else None,
        )
        result['wall_time_s'] = round(time.time() - start, 1)
        grid_results.append({'name': exp.name, **result})
        print(f'  Done in {result["wall_time_s"]}s, loss={result.get("final_loss", "?")}')
    except Exception as exc:
        print(f'  FAILED: {exc}')
        grid_results.append({'name': exp.name, 'error': str(exc)})

    finally:
        with (RESULTS_DIR / 'grid_results.json').open('w', encoding='utf-8') as handle:
            json.dump(grid_results, handle, indent=2, ensure_ascii=False)
print('\nGrid search complete.')


## Quick Eval on 300 Dev Questions

In [ ]:
from benchmarks.bird_loader import load_bird_dev, get_db_path, get_schema
from benchmarks.eval_ex import execution_match
from core.sql_extractor import extract_sql
from vllm import LLM, SamplingParams
from training.prompt_template import SYSTEM_MESSAGE
from core.prompt_builder import build_prompt

BIRD_DIR = DATA_DIR / 'bird'
all_bird_dev = load_bird_dev(BIRD_DIR)
eval_limit = int(os.environ.get('BIRD_LIMIT', '300'))
bird_dev = all_bird_dev if eval_limit == 0 else all_bird_dev[:eval_limit]

successful = [r for r in grid_results if 'error' not in r and r.get('adapter_path')]
successful.sort(key=lambda r: r.get('final_loss') if r.get('final_loss') is not None else float('inf'))
top_result = successful[0] if successful else None
top_adapter = top_result.get('adapter_path') if top_result else None
print(f'Evaluating best adapter: {top_adapter}')


In [ ]:
if top_result:
    adapter_path = Path(top_result['adapter_path'])
    merge_path = RESULTS_DIR / top_result['name'] / 'merged'
    merge_and_save(
        top_result['config']['model_id'],
        adapter_path,
        merge_path,
        load_in_4bit=top_result['config'].get('quantization') == '4bit',
    )
    with (RESULTS_DIR / 'best_checkpoint.json').open('w', encoding='utf-8') as handle:
        json.dump({
            'experiment': top_result['name'],
            'adapter_path': str(adapter_path),
            'merged_path': str(merge_path),
            'model_id': top_result['config']['model_id'],
        }, handle, indent=2)

    llm = LLM(model=str(merge_path), dtype='float16', max_model_len=2048, gpu_memory_utilization=0.85)
    tokenizer = llm.get_tokenizer()
    params = SamplingParams(temperature=0.0, max_tokens=512)

    correct = 0
    for record in bird_dev:
        schema = get_schema(BIRD_DIR, record['db_id'])
        messages = [
            {'role': 'system', 'content': SYSTEM_MESSAGE},
            {'role': 'user', 'content': build_prompt(record['question'], schema, record.get('evidence', ''))},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        output = llm.generate([prompt], params)[0]
        try:
            pred_sql = extract_sql(output.outputs[0].text)
        except ValueError:
            pred_sql = ''
        db_path = str(get_db_path(BIRD_DIR, record['db_id']))
        correct += int(execution_match(pred_sql, record['gold_sql'], db_path))

    print(f'EX on {len(bird_dev)} dev: {correct}/{len(bird_dev)} = {correct/len(bird_dev):.4f}')
else:
    print('No successful adapter to evaluate')


## Full Training (30-50K samples)

Run with the best config from the grid search above.

In [ ]:
# Optional second pass after selecting a configuration and confirming GPU budget.
# Set TRAIN_DATA=data/sft_subsets/sft_30000.jsonl and MAX_EXPERIMENTS=1, then rerun.
print('Use the same train(...) call above with a larger TRAIN_DATA file for full training.')


In [ ]:
# Optional: publish one adapter/merged checkpoint to a private Hugging Face model repo.
# In Kaggle, add HF_TOKEN and HF_REPO_ID as Secrets; values are never printed.
load_kaggle_secrets()
checkpoint_raw = os.environ.get('CHECKPOINT_DIR', '')
if not checkpoint_raw and 'merge_path' in globals() and Path(merge_path).is_dir():
    checkpoint_raw = str(merge_path)
if checkpoint_raw:
    print(publish_checkpoint(checkpoint_raw))
else:
    print('No checkpoint selected. Set CHECKPOINT_DIR before publishing.')